# 문제 6 — 좌표 변환 체인과 회전축 복원

로봇 팔이 카메라가 본 물체를 집으려면 **camera → link → base** 로 이어지는
변환을 타고 내려와야 합니다. TF2 가 현장에서 해 주는 일을 직접 구현합니다.

$$T^{base}_{cam} = T^{base}_{link}\,T^{link}_{cam},\qquad
\mathbf{p}_{base}=T^{base}_{cam}\begin{bmatrix}\mathbf{p}_{cam}\\1\end{bmatrix}$$

윗첨자/아랫첨자가 **이웃끼리 상쇄**되도록 곱하면 순서를 틀리지 않습니다.

## 이 노트북에서 해야 할 일

| # | 할 일 | 구현할 함수 |
|---|---|---|
| 6-1 | base→link, link→camera 두 변환을 정의하고 카메라 좌표를 base 로 바꾸기 | `CoordinateChain`, `default_chain`, `camera_point_to_base` |
| 6-2 | **왕복 검증** + 점군을 **반복문 없이** 한 번에 변환, 점 개수별 왕복 오차 그래프 | `transform`, `base_point_to_camera` |
| 6-3 | base·camera 좌표계와 변환된 점군을 **3D 로 함께 시각화** | — |
| 6-4 | **고유값 분해로 회전축 복원**, 대각합으로 회전각, 축 불변 확인 | `axis_angle_from_matrix` |
| 6-5 | 단위 쿼터니언을 만들어 SciPy 와 비교하고 **부호가 반대로 나올 수 있는 이유** 설명 | `quaternion_from_axis_angle` |

> 완성한 체인은 `src/coordinate_chain.py` 로 내보냅니다.
> 모듈 ④ 미니 프로젝트에서 그대로 import 해 쓰므로 함수 이름을 바꾸지 마세요.

In [ ]:
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.spatial.transform import Rotation

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.coordinate_chain import (CoordinateChain, base_point_to_camera,
                                  camera_point_to_base, default_chain)
from src.rotation import (axis_angle_from_matrix, quaternion_from_axis_angle, rodrigues,
                          rot_x, rot_y, rot_z)
from src.transform import inv_T, make_T, transform_points
from src.vectors import det, normalize

rng = np.random.default_rng(42)
np.set_printoptions(precision=6, suppress=True)

for _f in ["Malgun Gothic", "AppleGothic", "NanumGothic", "DejaVu Sans"]:
    if _f in {f.name for f in __import__("matplotlib").font_manager.fontManager.ttflist}:
        plt.rcParams["font.family"] = _f
        break
plt.rcParams["axes.unicode_minus"] = False


def check(label, condition):
    tag = "PASS" if condition else "FAIL"
    print("[" + tag + "] " + label)
    return bool(condition)


def draw_frame(ax, T, scale=0.15, name="", alpha=1.0):
    """동차변환 T 가 나타내는 좌표계를 그린다. (그대로 쓰면 됩니다)"""
    o = T[:3, 3]
    for i, c in enumerate(["r", "g", "b"]):
        ax.quiver(*o, *(T[:3, i] * scale), color=c, alpha=alpha, arrow_length_ratio=0.2)
    if name:
        ax.text(*(o + 0.03), name, fontsize=10, weight="bold")


print("SciPy Rotation 사용 가능")

## 6-1. 체인 구성과 카메라 → base 변환

`src/coordinate_chain.py` 의 `CoordinateChain` 은 부모-자식 관계를 등록해 두면
임의의 두 프레임 사이 변환을 알아서 조립합니다 (TF2 의 축소판).

지시문은 '임의의 회전·병진'을 쓰라고 하지만, 채점 수치를 맞추기 위해
아래 값을 권장합니다 (`default_chain` 의 docstring 과 같은 값).

- `base → link` : z축 30도 회전 후 (0.30, 0.00, 0.40) m 이동
- `link → camera` : y축 -20도, x축 90도 회전 후 (0.10, 0.05, 0.15) m 이동

문제 2 의 `rot_*` 와 문제 5 의 `make_T`/`inv_T` 를 그대로 재사용합니다.

**할 일** — `CoordinateChain` 의 `_path_to_root`, `T_from_root`, `T`, `transform` 과
`default_chain`, `camera_point_to_base` 를 구현하고 아래를 확인하세요.

In [ ]:
chain = default_chain()

# TODO: T(base<-link), T(link<-camera), T(base<-camera) 를 출력하고
#       체인이 조립한 결과가 두 행렬의 곱과 같은지 확인하세요.

In [ ]:
p_cam = np.array([0.20, -0.05, 0.60])       # 카메라가 본 물체 (60 cm 앞)

# TODO: camera_point_to_base 로 base 기준 좌표를 구해 출력하세요.
# TODO: base 원점에서의 거리, 카메라 원점의 base 기준 위치도 출력하세요.

In [ ]:
# --- 검증 ---
# TODO: 아래 항목을 check(...) 로 검증하세요.
#   - 체인 조립 결과가 행렬 곱과 일치하는가
#   - 합성 변환이 유효한 동차변환인가 (회전부 det = 1, 직교)
#   - 카메라 원점 (0,0,0) 이 base 에서 T 의 병진 성분으로 가는가
#   - 역방향 체인 T(camera<-base) == inv_T(T(base<-camera)) 인가
#   - 변환이 두 점 사이 거리를 보존하는가

## 6-2. 왕복 검증과 점군 벡터화

base 로 바꾼 좌표를 다시 카메라 기준으로 되돌리면 원래 값이 나와야 합니다.
수학적으로는 $T^{-1}T=I$ 라 당연하지만, 부동소수점에서는 아주 작은 오차가 남습니다.

점이 여러 개일 때는 반복문 대신 **한 번의 행렬 곱**으로 처리합니다.

$$P_{base}=P_{cam,h}\,T^{\mathsf{T}}\qquad (N\times 4)\cdot(4\times 4)$$

$(T P^{\mathsf{T}})^{\mathsf{T}}$ 대신 $P T^{\mathsf{T}}$ 를 쓰는 편이 유리한 이유도 생각해 보세요.

**할 일**

- 단일 점의 왕복 오차를 출력하세요.
- (N,3) 점군을 한 번에 변환하고, 반복문 결과와 일치하는지 확인하세요.
- 점 개수를 1 → 100만 까지 늘려 가며 **왕복 오차(최대·RMS)와 소요 시간**을 표로 출력하고,
  로그-로그 그래프 2개(오차 / 시간)로 그리세요.
- 점 개수가 늘어도 오차가 **누적되지 않는 이유**를 적으세요.

### 관찰과 해석

- 왕복 오차가 점 개수에 따라 어떻게 변하는가: `___`
- 그 이유: `___`
- 벡터화가 반복문보다 빠른 이유: `___`

In [ ]:
# TODO: 단일 점 왕복 검증 (p_cam -> base -> camera) 오차를 출력하세요.

In [ ]:
# TODO: (1000, 3) 점군을 만들어 한 번에 변환하고,
#       반복문 결과와 일치하는지 / 왕복 최대 오차를 출력하세요.

In [ ]:
counts = [1, 10, 100, 1_000, 10_000, 100_000, 1_000_000]

# TODO: 점 개수별로 왕복 오차(최대/RMS)와 벡터화·반복문 소요 시간을 재서 표로 출력하세요.
#       (반복문은 느리므로 1만 개까지만 재고 나머지는 np.nan 으로 두면 됩니다)

In [ ]:
# TODO: 그래프 2개를 그리세요.
#   왼쪽  : 점 개수별 왕복 오차 (loglog, 기계정밀도 eps 기준선 추가)
#   오른쪽: 벡터화 vs 반복문 소요 시간 (loglog)

In [ ]:
# --- 검증 ---
# TODO: 아래 항목을 check(...) 로 검증하세요.
#   - 단일 점 왕복 오차가 기계정밀도 수준인가
#   - 점군 왕복 오차도 기계정밀도 수준인가
#   - 오차가 점 개수에 따라 누적되지 않는가
#   - 벡터화 결과 == 반복문 결과
#   - 1만 개에서 벡터화가 반복문보다 빠른가
#   - 변환이 점 사이 거리를 보존하는가

## 6-3. 좌표계와 점군 3D 시각화

base·link·camera 세 좌표계와, 카메라가 본 점군을 base 기준으로 옮긴 결과를 함께 그립니다.
점군이 **카메라 앞쪽(카메라 z축 방향)** 에 놓여야 기하적으로 타당합니다.

**할 일**

- 카메라 기준으로 z축 앞 0.7 m 근처에 점군 300개를 만드세요.
- 왼쪽: 카메라 기준 점군, 오른쪽: base 기준 점군 + 세 좌표계 를 2분할로 그리세요.
- 체인 연결선(base → link → camera)과 카메라 시선(z축)을 함께 그리면 판단하기 쉽습니다.

In [ ]:
# TODO: 점군을 만들고 base 기준으로 변환하세요.

In [ ]:
# TODO: 2분할 3D 그림을 그리세요.

In [ ]:
# --- 검증 ---
# TODO: 아래 항목을 check(...) 로 검증하세요.
#   - 점군 중심이 카메라 z축(시선) 방향에 있는가 (내적으로 확인)
#   - 점군 중심까지 거리가 약 0.7 m 인가
#   - 점군의 퍼짐(분산)이 변환 후에도 보존되는가
#   - link 원점이 base 에서 기대한 위치에 있는가

## 6-4. 고유값 분해로 회전축 복원

**오일러 회전 정리**: 3차원의 모든 회전은 어떤 축 하나를 중심으로 한 회전이다.

회전축 $\mathbf{k}$ 는 회전에 의해 변하지 않는 방향이므로

$$R\mathbf{k}=\mathbf{k}=1\cdot\mathbf{k}$$

즉 **고유값 1 에 대응하는 고유벡터**입니다.
회전행렬의 고유값 세 개가 각각 어떤 값인지 직접 출력해 확인해 보세요.

회전각은 대각합에서 나옵니다. 고유값의 합 = 대각합이라는 사실에서
$\mathrm{tr}(R)$ 와 $\theta$ 의 관계를 직접 유도하세요.

**주의할 점 두 가지**

- `arccos` 의 치역이 $[0,\pi]$ 라 **어느 쪽으로 도는지**는 알 수 없습니다.
- 고유벡터는 **부호가 정해지지 않습니다**.

이 둘을 어떻게 해결할지 정하고(힌트: $R-R^{\mathsf{T}}$ 를 전개해 보세요) 구현하세요.
$\theta=0$ 과 $\theta=\pi$ 는 따로 처리해야 합니다 — 각각 왜 그런지도 적으세요.

### 유도와 규약

- trace 와 회전각의 관계: `___`
- 축의 부호를 정하는 방법: `___`
- theta = 0 / theta = pi 처리: `___`

In [ ]:
# TODO: R_chain = T(base<-camera) 의 회전 부분을 꺼내
#       np.linalg.eig 로 고유값 세 개를 출력하고, 고유값 1 의 고유벡터를 확인하세요.

In [ ]:
# TODO: axis_angle_from_matrix 로 축과 각을 복원해 출력하세요 (도 단위도 함께).
# TODO: trace(R) 로 구한 cos(theta) 도 출력해 비교하세요.
# TODO: 축 불변 검증 — R @ axis 와 axis 를 비교하세요.
# TODO: 복원한 축/각으로 rodrigues 를 돌려 원래 R 이 나오는지 확인하세요.

In [ ]:
# TODO: (권장) 3D 그림 하나 — base 좌표계, 복원한 회전축(직선),
#       축 둘레를 theta 만큼 도는 호(arc)를 그려 '축 + 각' 을 시각화하세요.

In [ ]:
# --- 검증 ---
# TODO: 아래 항목을 check(...) 로 검증하세요.
#   - 고유값 중 하나가 1 인가 / 세 고유값의 절댓값이 모두 1 인가
#   - 복원한 축이 단위벡터인가
#   - R @ axis == axis (축 불변), R.T @ axis == axis 인가
#   - trace 로 구한 각이 고유값의 위상과 일치하는가
#   - 복원한 축/각으로 R 을 재구성할 수 있는가
#   - SciPy 의 회전벡터 크기와 각이 일치하는가 (# 비교 대상)

## 6-5. 단위 쿼터니언과 SciPy 비교 — 부호는 왜 반대로 나올 수 있나

축 $\mathbf{k}$, 각 $\theta$ 에서 단위 쿼터니언은

$$q=\left(\mathbf{k}\sin\tfrac{\theta}{2},\;\cos\tfrac{\theta}{2}\right)\quad (x,y,z,w)$$

**할 일**

- 복원한 축·각으로 쿼터니언을 만들고 `Rotation.from_matrix(R).as_quat()` 와 비교하세요.
- 부호까지 같은지 / 부호를 무시하면 같은지 각각 확인하고, $|q \cdot q_{ref}|$ 도 보세요.
- $q$ 와 $-q$ 를 각각 회전행렬로 되돌려 보고, 무엇을 알 수 있는지 적으세요.
- **축을 뒤집고 각을 $2\pi-\theta$ 로 바꾸면** 어떤 쿼터니언이 나오는지 확인하세요.
- 위 관찰을 모아 **부호가 반대로 나올 수 있는 이유**를 설명하고,
  실무에서 비교·보간할 때 어떻게 다뤄야 하는지 적으세요.

### 부호 차이의 이유와 실무 대처

- 관찰: `___`
- 이유: `___`
- 비교할 때: `___`
- 보간(SLERP)·필터에 쓸 때: `___`

In [ ]:
# TODO: q_mine = quaternion_from_axis_angle(axis, angle) 와
#       q_scipy = Rotation.from_matrix(R_chain).as_quat() 를 비교 출력하세요.
# TODO: q 와 -q 를 각각 회전행렬로 되돌려 원본과의 오차를 출력하세요.
# TODO: 축을 뒤집고 각을 2pi - theta 로 바꾼 쿼터니언도 만들어 비교하세요.

In [ ]:
# --- 검증 ---
# TODO: 아래 항목을 check(...) 로 검증하세요.
#   - 직접 계산한 쿼터니언이 단위 노름인가
#   - SciPy 결과와 부호를 무시하면 일치하는가
#   - |q · q_scipy| == 1 인가
#   - q 와 -q 로 복원한 R 이 모두 원본과 일치하는가
#   - 축 반전 + (2pi - theta) 가 -q 를 주는가
#   - 무작위 회전 100개에서 부호를 무시하면 항상 일치하는가

## 답안 템플릿 정리

In [ ]:
summary = """
1. 카메라 좌표에서 base 좌표로의 변환 결과
   p_cam  = ___  ->  p_base = ___
   사용한 체인: T(base<-camera) = ___

2. 왕복 검증: 단일 점 오차 ___, 점군 100만 개에서 최대 ___
   - 점 개수별 오차 그래프: 위 왼쪽 그림
   - 오차가 누적되지 않는 이유: ___
   - 벡터화 vs 반복문 속도 (1만 개): ___ 배

3. 좌표계와 점군 3D 시각화: 위 2분할 그림 참조
   - 기하적으로 타당하다고 판단한 근거: ___

4. 복원한 회전축: ___ / 회전각: ___ 도
   - 축 불변 검증: ___ (|R k - k| = ___)
   - 축과 각을 구한 방법: ___

5. 쿼터니언 비교 (x, y, z, w)
   - 직접 계산: ___
   - SciPy    : ___
   - 부호 무시하고 일치: ___,  |q · q_scipy| = ___
   - 부호 차이의 이유: ___
"""
print(summary)